In [ ]:
# nltk.download('punkt_tab')
# nltk.download('stopwords', quiet=True)
# nltk.download('wordnet', quiet=True)

In [ ]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
training_data = pd.read_csv("twitter_training.csv",header=None,names=["ID", "Topic", "Sentiment", "Tweet"])
training_data.head(4)

,ID,Topic,Sentiment,Tweet
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...


In [ ]:
testing_data = pd.read_csv("twitter_validation.csv",header=None,names=["ID", "Topic", "Sentiment", "Tweet"])
testing_data.head(4)

,ID,Topic,Sentiment,Tweet
0,3364,Facebook,Irrelevant,I mentioned on Facebook that I was struggling ...
1,352,Amazon,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
2,8312,Microsoft,Negative,@Microsoft Why do I pay for WORD when it funct...
3,4371,CS-GO,Negative,"CSGO matchmaking is so full of closet hacking,..."


In [ ]:
training_data.isnull().sum()

,0
ID,0
Topic,0
Sentiment,0
Tweet,686


In [ ]:
training_data.dropna(inplace=True)

In [ ]:
training_data.isnull().sum()

,0
ID,0
Topic,0
Sentiment,0
Tweet,0


In [ ]:
testing_data.isnull().sum()

,0
ID,0
Topic,0
Sentiment,0
Tweet,0


In [ ]:
# initialize lemmatizer and load stop words
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# I used a function, because I'll use the same preprocessing on every message I have
def preprocessing(text):
    if pd.isna(text):
      return ""
    # remove punctuation and marks using regex
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\d+', '', text) # I don't if I should remove numbers or not

    # convert to lowercase (normalization)
    text = text.lower()

    # tokenize the text using word tokenizer
    words = nltk.word_tokenize(text)

    # remove Stop words
    words = [word for word in words if word not in stop_words]

    # use lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]

    # Join the words back into a single string
    return " ".join(words)

In [ ]:
# apply preprocessing to all messages in the data
training_data['clean_tweets'] = training_data['Tweet'].apply(preprocessing)
training_data['clean_tweets'].head(5)

,clean_tweets
0,im getting borderland murder
1,coming border kill
2,im getting borderland kill
3,im coming borderland murder
4,im getting borderland murder


In [ ]:
testing_data['clean_tweets'] = testing_data['Tweet'].apply(preprocessing)
testing_data['clean_tweets'].head(5)

,clean_tweets
0,mentioned facebook struggling motivation go ru...
1,bbc news amazon bos jeff bezos reject claim co...
2,microsoft pay word function poorly samsungus c...
3,csgo matchmaking full closet hacking truly awf...
4,president slapping american face really commit...


In [ ]:
training_data['Sentiment'] = training_data['Sentiment'].str.strip().str.lower()
training_data['Sentiment'] = training_data['Sentiment'].map({'negative': 0,'neutral': 1,'positive': 2,'irrelevant': 3})

testing_data['Sentiment'] = testing_data['Sentiment'].str.strip().str.lower()
testing_data['Sentiment'] = testing_data['Sentiment'].map({'negative': 0,'neutral': 1,'positive': 2,'irrelevant': 3})

In [ ]:
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(training_data['clean_tweets'])
y_train = training_data['Sentiment']

X_test = vectorizer.transform(testing_data['clean_tweets'])
y_test = testing_data['Sentiment']

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (73996, 35633)
Testing shape: (1000, 35633)


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

# Predict
y_pred = rf_model.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.972
              precision    recall  f1-score   support

           0       0.96      0.97      0.97       266
           1       0.98      0.97      0.97       285
           2       0.97      0.97      0.97       277
           3       0.99      0.97      0.98       172

    accuracy                           0.97      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.97      0.97      0.97      1000

[[259   3   3   1]
 [  5 276   4   0]
 [  3   3 270   1]
 [  3   0   2 167]]
